In [1]:
from __future__ import annotations

import json
from pathlib import Path
from typing import Dict, TextIO, Tuple


# Map arXiv category strings -> (super_folder, subcategory_filename_stem)
# "super_folder" is what you want: physics, cs, math, ...
def categorize_arxiv(cat: str) -> Tuple[str, str]:
    """
    Returns:
      super: folder name (e.g. 'physics', 'cs', 'math', '_unknown')
      sub:   subcategory identifier for per-subcat file (e.g. 'hep-th', 'cs.CL')
    """
    cat = (cat or "").strip()
    if not cat:
        return ("_unknown", "_unknown")

    # Full category goes to per-subcat file (lowest category you asked for)
    sub = cat

    # Physics umbrella:
    # - primary class 'physics.*'
    # - legacy/high-energy/etc standalone classes like hep-th, gr-qc, quant-ph, ...
    # - astro-ph.*, cond-mat.*, nucl-*, hep-*
    physics_prefixes = (
        "physics.", "astro-ph.", "cond-mat.",
    )
    physics_exact_prefixes = (
        "hep-", "nucl-",
    )
    physics_exact = {
        "gr-qc", "quant-ph",
    }

    if cat.startswith(physics_prefixes) or cat.startswith(physics_exact_prefixes) or cat in physics_exact:
        return ("physics", sub)

    # Otherwise, group by the part before '.' (cs, math, stat, econ, eess, q-bio, q-fin, etc.)
    super_folder = cat.split(".", 1)[0]  # e.g. "cs" from "cs.CL"
    return (super_folder, sub)


def split_jsonl_by_super_and_subcategory(
    in_path: str | Path,
    out_dir: str | Path,
    *,
    unknown_bucket: str = "_unknown",
) -> None:
    """
    Input: JSONL (one JSON object per line) with key 'arxiv_primary_category'.
    Output structure:

      out_dir/<super>/<super>.jsonl                 (ALL records in that super group)
      out_dir/<super>/<sub>.jsonl                   (records for that exact subcategory)

    With categorize_arxiv(), physics-like categories are forced into out_dir/physics/.
    """
    in_path = Path(in_path)
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    writers: Dict[Path, TextIO] = {}

    def get_writer(path: Path) -> TextIO:
        if path not in writers:
            path.parent.mkdir(parents=True, exist_ok=True)
            writers[path] = path.open("w", encoding="utf-8", newline="\n")
        return writers[path]

    try:
        with in_path.open("r", encoding="utf-8") as f:
            for lineno, line in enumerate(f, start=1):
                line = line.strip()
                if not line:
                    continue

                try:
                    obj = json.loads(line)
                except json.JSONDecodeError as e:
                    raise ValueError(f"Bad JSON on line {lineno}: {e}") from e

                cat = (obj.get("arxiv_primary_category") or "").strip()
                if not cat:
                    cat = unknown_bucket

                super_folder, sub = categorize_arxiv(cat)
                if not super_folder:
                    super_folder = unknown_bucket
                if not sub:
                    sub = unknown_bucket

                super_dir = out_dir / super_folder

                # 1) Write to the "ALL records in this super category" file
                all_path = super_dir / f"{super_folder}.jsonl"
                get_writer(all_path).write(json.dumps(obj, ensure_ascii=False) + "\n")

                # 2) Write to the per-subcategory file (within the same folder)
                sub_path = super_dir / f"{sub}.jsonl"
                get_writer(sub_path).write(json.dumps(obj, ensure_ascii=False) + "\n")

    finally:
        for fp in writers.values():
            fp.close()


# Example usage:
split_jsonl_by_super_and_subcategory("out/equations.jsonl", "SymbolicPriors")